# 🐾 Animal Image Classification

A PyTorch image classification project that recognizes **10 animal categories** using CNN and Transfer Learning.

**Models compared:**
- `AdvancedCNN` — custom deep CNN, trained from scratch
- `ResNet18` — Transfer Learning, pretrained on ImageNet
- `MobileNetV2` — Transfer Learning, pretrained on ImageNet ✅ Best

**Categories:** `butterfly` · `cat` · `chicken` · `cow` · `dog` · `elephant` · `horse` · `sheep` · `spider` · `squirrel`

---
**Notebook sections:**
1. Libraries
2. Dataset & EDA
3. Model Definitions
4. Training — all 3 models
5. Comparison — curves, accuracy, confusion matrix
6. Inference

## 1. Libraries

In [ ]:
!pip install torch torchvision opencv-python Pillow scikit-learn matplotlib tqdm tensorboard -q

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToTensor, Compose, Resize, ColorJitter, RandomAffine
from torchvision.models import (
    resnet18, ResNet18_Weights,
    mobilenet_v2, MobileNet_V2_Weights
)
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from PIL import Image
from tqdm.notebook import tqdm
from collections import Counter
import matplotlib.pyplot as plt
import cv2
import shutil
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {device}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')

## 2. Dataset & EDA
*(Source: `datasets.py` — `CNN_datasets_20Nov.py`)*

In [ ]:
# ── AnimalDataset (from CNN_datasets_20Nov.py) ────────────────────────────────
class AnimalDataset(Dataset):
    def __init__(self, root, is_train, transform=None):
        self.categories = ["butterfly", "cat", "chicken", "cow", "dog",
                           "elephant", "horse", "sheep", "spider", "squirrel"]
        if is_train:
            data_path = os.path.join(root, "train")
        else:
            data_path = os.path.join(root, "test")

        self.transform = transform
        self.images = []
        self.labels = []

        for idx, category in enumerate(self.categories):
            category_path = os.path.join(data_path, category)
            for item in os.listdir(category_path):
                image_path = os.path.join(category_path, item)
                image = cv2.imread(image_path)
                self.images.append(image_path)
                self.labels.append(idx)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        return image, label

print('AnimalDataset defined.')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH      = 'animals'       # ← change to your dataset path
IMAGE_SIZE     = 224
BATCH_SIZE     = 16
NUM_EPOCHS     = 100
LR             = 0.01
EARLY_STOP     = 3
CHECKPOINT_DIR = 'train_models'
LOG_DIR        = 'tensorboard'

# Train transform — with augmentation (from CNN_train_Animal_MobilenetV2_25Nov.py)
train_transform = Compose([
    ColorJitter(brightness=0.5, contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=(-0.2, 0.2)),
    RandomAffine(degrees=20, translate=(0, 0), scale=(0.85, 1.15), shear=(-30, 30)),
    ToTensor(),
    Resize((IMAGE_SIZE, IMAGE_SIZE))
])

# Val transform — no augmentation
val_transform = Compose([
    ToTensor(),
    Resize((IMAGE_SIZE, IMAGE_SIZE))
])

train_dataset = AnimalDataset(root=DATA_PATH, is_train=True,  transform=train_transform)
val_dataset   = AnimalDataset(root=DATA_PATH, is_train=False, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=4, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=4, drop_last=False)

CATEGORIES = train_dataset.categories
print(f'Train : {len(train_dataset)} images')
print(f'Val   : {len(val_dataset)} images')
print(f'Classes ({len(CATEGORIES)}): {CATEGORIES}')

In [ ]:
# ── EDA: Class distribution ───────────────────────────────────────────────────
train_counts = Counter(train_dataset.labels)
val_counts   = Counter(val_dataset.labels)

x = np.arange(len(CATEGORIES))
w = 0.35
fig, ax = plt.subplots(figsize=(13, 5))
b1 = ax.bar(x - w/2, [train_counts[i] for i in range(10)], w, label='Train', color='steelblue')
b2 = ax.bar(x + w/2, [val_counts[i]   for i in range(10)], w, label='Test',  color='coral')
ax.set_xticks(x)
ax.set_xticklabels([c.capitalize() for c in CATEGORIES], rotation=30, ha='right')
ax.set_ylabel('Number of images')
ax.set_title('Class Distribution — Train vs Test')
ax.legend()
ax.bar_label(b1, padding=3, fontsize=8)
ax.bar_label(b2, padding=3, fontsize=8)
plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=150)
plt.show()

In [ ]:
# ── EDA: Sample images ────────────────────────────────────────────────────────
shown = {}
for i in range(len(train_dataset)):
    _, lbl = train_dataset[i]
    if lbl not in shown:
        shown[lbl] = i
    if len(shown) == 10:
        break

fig, axes = plt.subplots(2, 5, figsize=(15, 7))
fig.suptitle('Sample Images — One Per Class', fontsize=14, fontweight='bold')
for cls_idx in range(10):
    row, col = divmod(cls_idx, 5)
    img = Image.open(train_dataset.images[shown[cls_idx]]).convert('RGB')
    axes[row][col].imshow(img)
    axes[row][col].set_title(CATEGORIES[cls_idx].capitalize(), fontsize=11)
    axes[row][col].axis('off')
plt.tight_layout()
plt.savefig('eda_sample_images.png', dpi=150)
plt.show()

## 3. Model Definitions
*(Source: `models.py` — `CNN_models_20Nov.py`)*

| Model | Type | Pretrained | ~Params |
|-------|------|-----------|--------|
| AdvancedCNN | Custom CNN | ❌ From scratch | 3.2M |
| ResNet18 | Transfer Learning | ✅ ImageNet | 11.2M |
| MobileNetV2 | Transfer Learning | ✅ ImageNet | 3.4M |

In [ ]:
# ── Models (from CNN_models_20Nov.py) ─────────────────────────────────────────
class MyNeuralNetwork(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Sequential(nn.Linear(3072, 8),  nn.ReLU())
        self.fc2 = nn.Sequential(nn.Linear(8,   16),  nn.ReLU())
        self.fc3 = nn.Sequential(nn.Linear(16,  32),  nn.ReLU())
        self.fc4 = nn.Sequential(nn.Linear(32,  64),  nn.ReLU())
        self.fc5 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x); x = self.fc2(x)
        x = self.fc3(x); x = self.fc4(x)
        return self.fc5(x)


class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.fc1 = nn.Sequential(nn.Dropout(0.5), nn.Linear(2048, 512), nn.ReLU())
        self.fc2 = nn.Sequential(nn.Dropout(0.5), nn.Linear(512,  128), nn.ReLU())
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv2(self.conv1(x))
        x = x.view(x.shape[0], -1)
        return self.fc3(self.fc2(self.fc1(x)))


class AdvancedCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = self._make_block(3,  8,  3)
        self.conv2 = self._make_block(8,  16, 3)
        self.conv3 = self._make_block(16, 32, 3)
        self.conv4 = self._make_block(32, 64, 3)
        self.conv5 = self._make_block(64, 64, 3)
        self.fc1 = nn.Sequential(nn.Dropout(0.5), nn.Linear(64*7*7, 512), nn.ReLU())
        self.fc2 = nn.Sequential(nn.Dropout(0.5), nn.Linear(512, 128), nn.ReLU())
        self.fc3 = nn.Linear(128, num_classes)

    def _make_block(self, in_c, out_c, k):
        return nn.Sequential(
            nn.Conv2d(in_c,  out_c, k, padding='same'), nn.BatchNorm2d(out_c), nn.ReLU(),
            nn.Conv2d(out_c, out_c, k, padding='same'), nn.BatchNorm2d(out_c), nn.ReLU(),
            nn.MaxPool2d(2)
        )

    def forward(self, x):
        for conv in [self.conv1, self.conv2, self.conv3, self.conv4, self.conv5]:
            x = conv(x)
        x = x.view(x.shape[0], -1)
        return self.fc3(self.fc2(self.fc1(x)))


# ── Param count ───────────────────────────────────────────────────────────────
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

m_adv = AdvancedCNN(10)
m_r18 = resnet18(weights=ResNet18_Weights.DEFAULT)
m_r18.fc = nn.Linear(512, 10)
m_mob = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
m_mob.classifier[1] = nn.Linear(1280, 10)

names  = ['AdvancedCNN\n(scratch)', 'ResNet18\n(pretrained)', 'MobileNetV2\n(pretrained)']
params = [count_params(m_adv), count_params(m_r18), count_params(m_mob)]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(names, [p/1e6 for p in params],
              color=['#e74c3c','#3498db','#2ecc71'], edgecolor='white', width=0.5)
ax.bar_label(bars, labels=[f'{p/1e6:.2f}M' for p in params], padding=5, fontsize=11)
ax.set_ylabel('Parameters (Millions)')
ax.set_title('Trainable Parameters per Model')
ax.set_ylim(0, max(params)/1e6 * 1.3)
plt.tight_layout()
plt.savefig('model_params.png', dpi=150)
plt.show()
for n, p in zip(names, params):
    print(f'{n.replace(chr(10)," "):30s}: {p:>12,} params')

## 4. Training
*(Source: `train.py` — `CNN_train_Animal_MobilenetV2_25Nov.py`)*

In [ ]:
# ── plot_confusion_matrix (from train.py) ─────────────────────────────────────
def plot_confusion_matrix(writer, cm, class_names, epoch):
    figure = plt.figure(figsize=(8, 8))
    plt.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45, ha='right')
    plt.yticks(tick_marks, class_names)
    cm_norm = np.around(cm.astype('float') / cm.sum(axis=1)[:, np.newaxis], decimals=2)
    threshold = cm_norm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = 'white' if cm_norm[i, j] > threshold else 'black'
            plt.text(j, i, cm_norm[i, j], horizontalalignment='center', color=color)
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    if writer:
        writer.add_figure('confusion_matrix', figure, epoch)
    return figure

In [ ]:
# ── Reusable training function (mirrors train.py exactly) ─────────────────────
def train_model(model, model_name, train_loader, val_loader,
                num_epochs=NUM_EPOCHS, lr=LR, early_stop=EARLY_STOP,
                checkpoint_dir=CHECKPOINT_DIR, log_dir=LOG_DIR):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    ckpt_dir = os.path.join(checkpoint_dir, model_name)
    if os.path.isdir(ckpt_dir):
        shutil.rmtree(ckpt_dir)
    os.makedirs(ckpt_dir, exist_ok=True)

    log_path = os.path.join(log_dir, model_name)
    os.makedirs(log_path, exist_ok=True)
    writer = SummaryWriter(log_path)

    best_accuracy = -1
    best_epoch    = 0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        # ── Train ────────────────────────────────────────────────────────────
        model.train()
        train_loss = []
        bar = tqdm(train_loader, colour='cyan',
                   desc=f'[{model_name}] Epoch {epoch+1}/{num_epochs}')
        for it, (images, labels) in enumerate(bar):
            images, labels = images.to(device), labels.to(device)
            predictions    = model(images)
            loss_value     = criterion(predictions, labels)
            optimizer.zero_grad()
            loss_value.backward()
            optimizer.step()
            bar.set_description(
                f'[{model_name}] Epoch {epoch+1}/{num_epochs}. Loss {loss_value.item():.4f}'
            )
            train_loss.append(loss_value.item())
            writer.add_scalar('Train/Loss', np.mean(train_loss),
                              global_step=epoch * len(train_loader) + it)
        history['train_loss'].append(np.mean(train_loss))

        # ── Validate ─────────────────────────────────────────────────────────
        model.eval()
        all_labels, all_predictions, all_loss = [], [], []
        with torch.no_grad():
            bar_val = tqdm(val_loader, colour='green',
                           desc=f'[{model_name}] Validating')
            for images, labels in bar_val:
                images, labels = images.to(device), labels.to(device)
                outputs     = model(images)
                loss_value  = criterion(outputs, labels)
                predictions = torch.argmax(outputs, dim=1)
                all_labels.extend(labels.tolist())
                all_predictions.extend(predictions.tolist())
                all_loss.append(loss_value.item())

        accuracy  = accuracy_score(all_labels, all_predictions)
        conf_matrix = confusion_matrix(all_labels, all_predictions)
        plot_confusion_matrix(writer, conf_matrix, CATEGORIES, epoch)
        plt.close('all')
        avg_loss  = np.mean(all_loss)
        history['val_loss'].append(avg_loss)
        history['val_acc'].append(accuracy)

        print(f'Epoch {epoch+1}/{num_epochs}. Loss {avg_loss:.4f}. Accuracy {accuracy:.4f}')
        writer.add_scalar('Val/Loss',     avg_loss, global_step=epoch)
        writer.add_scalar('Val/Accuracy', accuracy, global_step=epoch)

        checkpoint = {
            'model':         model.state_dict(),
            'optimizer':     optimizer.state_dict(),
            'epoch':         epoch + 1,
            'best_epoch':    best_epoch,
            'best_accuracy': best_accuracy,
        }
        torch.save(checkpoint, os.path.join(ckpt_dir, 'last.pt'))
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_epoch    = epoch
            torch.save(checkpoint, os.path.join(ckpt_dir, 'best.pt'))
            print(f'  ✅ New best — Acc: {best_accuracy:.4f}')

        if epoch - best_epoch > early_stop:
            print(f'Stop training process at epoch {epoch+1}')
            break

    history['best_accuracy'] = best_accuracy
    history['best_epoch']    = best_epoch
    history['all_labels']    = all_labels
    history['all_preds']     = all_predictions
    writer.close()
    print(f'\n[{model_name}] Best Val Acc: {best_accuracy:.4f} at epoch {best_epoch+1}')
    return history

### 4.1 AdvancedCNN — from scratch

In [ ]:
model_advcnn = AdvancedCNN(num_classes=len(CATEGORIES))
hist_advcnn  = train_model(model_advcnn, 'AdvancedCNN', train_loader, val_loader)

### 4.2 ResNet18 — Transfer Learning

In [ ]:
model_resnet = resnet18(weights=ResNet18_Weights.DEFAULT)
model_resnet.fc = nn.Linear(in_features=512, out_features=len(CATEGORIES), bias=True)
hist_resnet  = train_model(model_resnet, 'ResNet18', train_loader, val_loader)

### 4.3 MobileNetV2 — Transfer Learning ✅ Best

In [ ]:
model_mobilenet = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
model_mobilenet.classifier[1] = nn.Linear(in_features=1280, out_features=len(CATEGORIES), bias=True)
hist_mobilenet  = train_model(model_mobilenet, 'MobileNetV2', train_loader, val_loader)

## 5. Comparison

In [ ]:
all_histories = {
    'AdvancedCNN (scratch)':    hist_advcnn,
    'ResNet18 (pretrained)':    hist_resnet,
    'MobileNetV2 (pretrained)': hist_mobilenet,
}
COLORS = {
    'AdvancedCNN (scratch)':    '#e74c3c',
    'ResNet18 (pretrained)':    '#3498db',
    'MobileNetV2 (pretrained)': '#2ecc71',
}

# Training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training Curves — All Models', fontsize=14, fontweight='bold')
for name, hist in all_histories.items():
    c = COLORS[name]
    axes[0].plot(hist['train_loss'], label=name, color=c)
    axes[1].plot(hist['val_loss'],   label=name, color=c)
    axes[2].plot(hist['val_acc'],    label=name, color=c)
for ax, title in zip(axes, ['Train Loss', 'Val Loss', 'Val Accuracy']):
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('comparison_training_curves.png', dpi=150)
plt.show()

In [ ]:
# Best accuracy bar chart
names_hist = list(all_histories.keys())
accs = [all_histories[n]['best_accuracy'] * 100 for n in names_hist]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(names_hist, accs, color=[COLORS[n] for n in names_hist],
              edgecolor='white', width=0.5)
ax.bar_label(bars, labels=[f'{a:.2f}%' for a in accs], padding=5,
             fontsize=11, fontweight='bold')
ax.set_ylabel('Best Validation Accuracy (%)')
ax.set_title('Best Validation Accuracy — All Models')
ax.set_ylim(0, 115)
plt.tight_layout()
plt.savefig('comparison_best_accuracy.png', dpi=150)
plt.show()

In [ ]:
# Per-class accuracy
def per_class_acc(labels, preds):
    cm = confusion_matrix(labels, preds)
    return cm.diagonal() / cm.sum(axis=1)

x = np.arange(len(CATEGORIES))
width = 0.28
fig, ax = plt.subplots(figsize=(14, 6))
for i, (name, hist) in enumerate(all_histories.items()):
    cls_accs = per_class_acc(hist['all_labels'], hist['all_preds'])
    ax.bar(x + (i-1)*width, cls_accs*100, width,
           label=name, color=list(COLORS.values())[i], alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([c.capitalize() for c in CATEGORIES], rotation=30, ha='right')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Per-Class Accuracy — All Models')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('comparison_per_class_accuracy.png', dpi=150)
plt.show()

In [ ]:
# Confusion matrix — MobileNetV2
cm = confusion_matrix(hist_mobilenet['all_labels'], hist_mobilenet['all_preds'])
fig = plot_confusion_matrix(None, cm, CATEGORIES, epoch=0)
plt.savefig('confusion_matrix_mobilenetv2.png', dpi=150, bbox_inches='tight')
plt.show()

# Classification report
print('Classification Report — MobileNetV2')
print('='*55)
print(classification_report(
    hist_mobilenet['all_labels'],
    hist_mobilenet['all_preds'],
    target_names=CATEGORIES
))

In [ ]:
# Summary table
print('\n' + '='*75)
print(f'  {"Model":<28} {"Pretrained":^12} {"Best Acc":^12} {"Best Epoch":^12} {"Min Loss":^12}')
print('='*75)
for name, hist in all_histories.items():
    pre = 'No' if 'scratch' in name else 'Yes'
    print(f"  {name:<28} {pre:^12} {hist['best_accuracy']*100:^11.2f}% "
          f"{hist['best_epoch']+1:^12} {min(hist['val_loss']):^12.4f}")
print('='*75)
best = max(all_histories, key=lambda n: all_histories[n]['best_accuracy'])
print(f'\n🏆 Best: {best} ({all_histories[best]["best_accuracy"]*100:.2f}%)')

## 6. Inference
*(Source: `inference_image.py` — `CNN_Inference_image25Nov.py`)*

In [ ]:
def predict_image(image_path, checkpoint_path='train_models/MobileNetV2/best.pt',
                  image_size=224):
    """
    Predict animal class for a single image.
    Preprocessing identical to CNN_Inference_image25Nov.py.
    """
    categories = ["butterfly", "cat", "chicken", "cow", "dog",
                  "elephant", "horse", "sheep", "spider", "squirrel"]
    model = mobilenet_v2(weights=None)
    model.classifier[1] = nn.Linear(1280, len(categories), bias=True)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model'])
    model.eval().to(device)

    # Same preprocessing as CNN_Inference_image25Nov.py
    ori_image = cv2.imread(image_path)
    image     = cv2.cvtColor(ori_image, cv2.COLOR_BGR2RGB)
    image     = cv2.resize(image, (image_size, image_size))
    image     = image / 255.
    image     = np.transpose(image, (2, 0, 1))
    image     = image[None, :, :, :]
    image     = torch.from_numpy(image).float().to(device)

    softmax = nn.Softmax()
    with torch.no_grad():
        output = model(image)
        probs  = softmax(output[0])
    predicted_prob, predicted_idx = torch.max(probs, dim=0)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(cv2.cvtColor(ori_image, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'{categories[predicted_idx]}: {predicted_prob*100:.2f}%',
                      fontsize=13, fontweight='bold')
    axes[0].axis('off')
    top5_probs, top5_idx = torch.topk(probs, 5)
    axes[1].barh([categories[i] for i in top5_idx.tolist()][::-1],
                 top5_probs.tolist()[::-1], color='steelblue')
    axes[1].set_xlabel('Probability')
    axes[1].set_title('Top-5 Predictions')
    plt.tight_layout()
    plt.show()
    return categories[predicted_idx], float(predicted_prob)

# Usage:
# label, conf = predict_image('path/to/image.jpg')
# print(f'{label}: {conf*100:.2f}%')
print('predict_image() defined.')

In [ ]:
# Show predictions on validation set
def show_val_predictions(model, dataset, n=10):
    model.eval().to(device)
    indices = np.random.choice(len(dataset), n, replace=False)
    fig, axes = plt.subplots(2, 5, figsize=(15, 7))
    fig.suptitle('Validation Predictions — MobileNetV2\nGreen=Correct | Red=Wrong',
                 fontsize=13, fontweight='bold')
    for ax, idx in zip(axes.flat, indices):
        img_tensor, true_lbl = dataset[idx]
        with torch.no_grad():
            out  = model(img_tensor.unsqueeze(0).to(device))
            pred = torch.argmax(out, dim=1).item()
        img_np = np.clip(img_tensor.permute(1, 2, 0).numpy(), 0, 1)
        ax.imshow(img_np)
        ax.set_title(
            f'True : {CATEGORIES[true_lbl]}\nPred : {CATEGORIES[pred]}',
            fontsize=9,
            color='green' if pred == true_lbl else 'red'
        )
        ax.axis('off')
    plt.tight_layout()
    plt.savefig('val_predictions.png', dpi=150)
    plt.show()

show_val_predictions(model_mobilenet, val_dataset, n=10)

## TensorBoard
```bash
tensorboard --logdir tensorboard
```
Open **http://localhost:6006**

In [ ]:
%load_ext tensorboard
%tensorboard --logdir tensorboard